In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install econml
!pip install dowhy
import joblib, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from econml.dml import CausalForestDML
import networkx as nx
from dowhy import CausalModel

cf_canonical = joblib.load('/content/drive/MyDrive/CausalMedia-GH/cf_canonical_model.pkl')
full = pd.read_csv('/content/drive/MyDrive/CausalMedia-GH/oulad_full_corpus.csv')
X_encoded_v2 = pd.read_csv('/content/drive/MyDrive/CausalMedia-GH/X_encoded_columns_reference.csv')

print(f"ATE check: {cf_canonical.ate(X_encoded_v2):.4f}  (should match 0.0048)")

T = full['oucontent_clicks'].values
Y = full['performance_gain'].values

confounder_cols = list(X_encoded_v2.columns)
df_refute = X_encoded_v2.copy()
df_refute['oucontent_clicks'] = T
df_refute['performance_gain'] = Y

G = nx.DiGraph()
for col in confounder_cols:
    G.add_edge(col, 'oucontent_clicks')
    G.add_edge(col, 'performance_gain')
G.add_edge('oucontent_clicks', 'performance_gain')

model = CausalModel(data=df_refute, treatment='oucontent_clicks', outcome='performance_gain',
                     graph=G, effect_modifiers=confounder_cols)
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)

estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.econml.dml.CausalForestDML",
    effect_modifiers=confounder_cols,
    method_params={"init_params": {
        'model_y': HistGradientBoostingRegressor(random_state=42),
        'model_t': HistGradientBoostingRegressor(random_state=42),
        'discrete_treatment': False, 'honest': True,
        'min_samples_leaf': 50, 'n_estimators': 500, 'cv': 5, 'random_state': 42
    }, "fit_params": {}}
)
print(f"DoWhy-wrapped estimate: {estimate.value:.4f}  (should also match 0.0048)")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 9.3 MB/s eta 0:00:0

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

DoWhy-wrapped estimate: 0.0048  (should also match 0.0048)


In [ ]:
# Cell 1
refute_placebo = model.refute_estimate(identified_estimand, estimate,
    method_name="placebo_treatment_refuter", placebo_type="permute", num_simulations=5)
print(refute_placebo)
with open('/content/drive/MyDrive/CausalMedia-GH/refute_1_placebo_v2.txt', 'w') as f:
    f.write(str(refute_placebo))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

Refute: Use a Placebo Treatment
Estimated effect:0.004786482048864249
New effect:-2.570601776404729e-05
p value:0.4498007457736204



In [ ]:
# Cell 2
refute_random_cause = model.refute_estimate(identified_estimand, estimate,
    method_name="random_common_cause", num_simulations=5)
print(refute_random_cause)
with open('/content/drive/MyDrive/CausalMedia-GH/refute_2_random_cause_v2.txt', 'w') as f:
    f.write(str(refute_random_cause))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

Refute: Add a random common cause
Estimated effect:0.004786482048864249
New effect:0.004796837208535323
p value:0.3880687385227338



In [ ]:
# Cell 3
refute_subset = model.refute_estimate(identified_estimand, estimate,
    method_name="data_subset_refuter", subset_fraction=0.8, num_simulations=5)
print(refute_subset)
with open('/content/drive/MyDrive/CausalMedia-GH/refute_3_subset_v2.txt', 'w') as f:
    f.write(str(refute_subset))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

Refute: Use a subset of data
Estimated effect:0.004786482048864249
New effect:0.004710802096447007
p value:0.30334793387875414



In [ ]:
# Cell 4
refute_bootstrap = model.refute_estimate(identified_estimand, estimate,
    method_name="bootstrap_refuter", num_simulations=5)
print(refute_bootstrap)
with open('/content/drive/MyDrive/CausalMedia-GH/refute_4_bootstrap_v2.txt', 'w') as f:
    f.write(str(refute_bootstrap))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

Refute: Bootstrap Sample Dataset
Estimated effect:0.004786482048864249
New effect:0.00277337298764442
p value:0.0



In [ ]:
refute_bootstrap_v3 = model.refute_estimate(identified_estimand, estimate,
    method_name="bootstrap_refuter", num_simulations=8)
print(refute_bootstrap_v3)
with open('/content/drive/MyDrive/CausalMedia-GH/refute_4_bootstrap_v3.txt', 'w') as f:
    f.write(str(refute_bootstrap_v3))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

Refute: Bootstrap Sample Dataset
Estimated effect:0.004786482048864249
New effect:0.0027864394242786013
p value:0.0



In [ ]:
refute_bootstrap_v3 = model.refute_estimate(identified_estimand, estimate,
    method_name="bootstrap_refuter", num_simulations=8)
print(refute_bootstrap_v3)
with open('/content/drive/MyDrive/CausalMedia-GH/refute_4_bootstrap_v3.txt', 'w') as f:
    f.write(str(refute_bootstrap_v3))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

Refute: Bootstrap Sample Dataset
Estimated effect:0.004786482048864249
New effect:0.0025838172831139194
p value:0.0



In [ ]:
from scipy.stats import kruskal, mannwhitneyu

full['cate_v2'] = cf_canonical.effect(X_encoded_v2)

# --- imd_band heterogeneity (never checked before) ---
imd_order = ['0-10%', '10-20', '20-30%', '30-40%', '40-50%', '50-60%',
             '60-70%', '70-80%', '80-90%', '90-100%', 'Missing']
present_levels = [lvl for lvl in imd_order if lvl in full['imd_band'].unique()]
print("imd_band levels actually present:", full['imd_band'].unique())

full['imd_band'] = pd.Categorical(full['imd_band'], categories=present_levels, ordered=True)
print("\nMean CATE by imd_band:")
print(full.groupby('imd_band', observed=True)['cate_v2'].agg(['mean', 'std', 'count']).reindex(present_levels))

groups_imd = [full.loc[full['imd_band']==lvl, 'cate_v2'].values for lvl in present_levels]
stat_imd, p_imd = kruskal(*[g for g in groups_imd if len(g) > 0])
k_imd = len([g for g in groups_imd if len(g) > 0])
eps_imd = (stat_imd - k_imd + 1) / (len(full) - k_imd)
print(f"\nKruskal-Wallis: H={stat_imd:.2f}, p={p_imd:.4g}, epsilon-squared={eps_imd:.5f}")

# --- highest_education re-check on CORRECTED model ---
edu_order = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent',
             'HE Qualification', 'Post Graduate Qualification']
full['highest_education'] = pd.Categorical(full['highest_education'], categories=edu_order, ordered=True)
print("\nMean CATE by highest_education (corrected model):")
print(full.groupby('highest_education', observed=True)['cate_v2'].agg(['mean', 'std', 'count']).reindex(edu_order))

groups_edu = [full.loc[full['highest_education']==lvl, 'cate_v2'].values for lvl in edu_order]
stat_edu, p_edu = kruskal(*groups_edu)
eps_edu = (stat_edu - 5 + 1) / (len(full) - 5)
print(f"\nKruskal-Wallis: H={stat_edu:.2f}, p={p_edu:.4g}, epsilon-squared={eps_edu:.5f}")

full[['id_student', 'code_module', 'code_presentation', 'cate_v2', 'imd_band', 'highest_education']].to_csv(
    '/content/drive/MyDrive/CausalMedia-GH/heterogeneity_checks_v2.csv', index=False)
print("\nSaved heterogeneity_checks_v2.csv")

imd_band levels actually present: <ArrowStringArray>
[ '80-90%', '90-100%',  '50-60%',  '40-50%',  '60-70%',  '20-30%',  '70-80%',
   '10-20',  '30-40%',       nan,   '0-10%']
Length: 11, dtype: str

Mean CATE by imd_band:
              mean       std  count
imd_band                           
0-10%     0.005067  0.002004   1438
10-20     0.005348  0.002121   1630
20-30%    0.004690  0.001973   1755
30-40%    0.004986  0.002050   1869
40-50%    0.004685  0.002038   1694
50-60%    0.004741  0.002076   1737
60-70%    0.004529  0.002049   1637
70-80%    0.004974  0.002188   1703
80-90%    0.004705  0.002275   1642
90-100%   0.004524  0.002240   1594

Kruskal-Wallis: H=337.42, p=2.943e-67, epsilon-squared=0.01875

Mean CATE by highest_education (corrected model):
                                 mean       std  count
highest_education                                     
No Formal quals              0.004758  0.001994    121
Lower Than A Level           0.005313  0.001975   5897
A Level or

In [ ]:
missing_count = full['imd_band'].isna().sum()
print(f"Students silently excluded from the imd_band check: {missing_count}")

imd_order_v2 = ['0-10%', '10-20', '20-30%', '30-40%', '40-50%', '50-60%',
                '60-70%', '70-80%', '80-90%', '90-100%', 'Missing']

# First, set the categories for the 'imd_band' column to include 'Missing'.
# This ensures 'Missing' is a valid category before we try to fill NaNs with it.
full['imd_band'] = pd.Categorical(full['imd_band'], categories=imd_order_v2, ordered=True)

# Now, fill NaN values with 'Missing'. This will now work as 'Missing' is a defined category.
full['imd_band'] = full['imd_band'].fillna('Missing')

print("\nMean CATE by imd_band (corrected, Missing category included):")
print(full.groupby('imd_band', observed=True)['cate_v2'].agg(['mean', 'std', 'count']).reindex(imd_order_v2))

groups_imd_v2 = [full.loc[full['imd_band']==lvl, 'cate_v2'].values for lvl in imd_order_v2]
stat_v2, p_v2 = kruskal(*groups_imd_v2)
eps_v2 = (stat_v2 - 11 + 1) / (len(full) - 11)
print(f"\nKruskal-Wallis (corrected): H={stat_v2:.2f}, p={p_v2:.4g}, epsilon-squared={eps_v2:.5f}")

Students silently excluded from the imd_band check: 830

Mean CATE by imd_band (corrected, Missing category included):
              mean       std  count
imd_band                           
0-10%     0.005067  0.002004   1438
10-20     0.005348  0.002121   1630
20-30%    0.004690  0.001973   1755
30-40%    0.004986  0.002050   1869
40-50%    0.004685  0.002038   1694
50-60%    0.004741  0.002076   1737
60-70%    0.004529  0.002049   1637
70-80%    0.004974  0.002188   1703
80-90%    0.004705  0.002275   1642
90-100%   0.004524  0.002240   1594
Missing   0.004041  0.002248    830

Kruskal-Wallis (corrected): H=442.06, p=1.033e-88, epsilon-squared=0.02466
